In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

In [ ]:
PROJECT_ROOT  = Path.cwd().parents[1]
DATA_PATH     = PROJECT_ROOT / "analysis" / "data" / "Lumiere_Country_Full_Backup.csv"
CLUSTERS_PATH = PROJECT_ROOT / "analysis" / "data" / "clustered_films_k6.csv"

In [ ]:
df_admissions = pd.read_csv(DATA_PATH)
df_clusters   = pd.read_csv(CLUSTERS_PATH)

budget_both = pd.read_excel(PROJECT_ROOT / "analysis" / "data" / "IMDB_financials.xlsx", sheet_name="Both")
budget_only = pd.read_excel(PROJECT_ROOT / "analysis" / "data" / "IMDB_financials.xlsx", sheet_name="Budget only")
df_budget = pd.concat([budget_both, budget_only], ignore_index=True)
df_budget = df_budget[["titleId", "budget_eur"]].dropna(subset=["budget_eur"])

assert len(df_admissions) == len(df_clusters), "Row count mismatch!"
df = df_admissions.copy()
df["cluster"] = df_clusters["cluster"].values
df = df.merge(df_budget, on="titleId", how="inner")

In [ ]:
def parse_country_admissions(value):
    if pd.isna(value) or value == "Not Found":
        return {}
    try:
        return json.loads(value.replace("'", '"'))
    except (json.JSONDecodeError, TypeError):
        return {}

records = []
for _, row in df.iterrows():
    admissions = parse_country_admissions(row["country_admissions"])
    for country, count in admissions.items():
        if country.startswith("Total"):
            continue
        records.append({
            "titleId":       row["titleId"],
            "originalTitle": row["originalTitle"],
            "plotMedium":    row["plotMedium"],
            "mainCountry":   row["mainCountry"],
            "firstLanguage": row["firstLanguage"],
            "genres":        row["genres"],
            "cluster":       row["cluster"],
            "country":       country,
            "admissions":    count,
        })

df_long = pd.DataFrame(records)

films_with_gbie = set(df_long[df_long["country"] == "GBIE"]["titleId"])
mask_redundant = (
    df_long["country"].isin(["GB", "IE"]) &
    df_long["titleId"].isin(films_with_gbie)
)
df_long = df_long[~mask_redundant].copy()

MIN_FILMS = 50
country_film_counts = df_long.groupby("country")["titleId"].nunique()
small_countries = country_film_counts[country_film_counts < MIN_FILMS].index.tolist()
df_long = df_long[~df_long["country"].isin(small_countries)].copy()

df_long["log_admissions"] = np.log1p(df_long["admissions"])
df_long["admissions_zscore"] = (
    df_long.groupby("country")["log_admissions"]
    .transform(lambda x: (x - x.mean()) / x.std())
)

In [ ]:
film_ids = df_long["titleId"].unique()
train_films, test_films = train_test_split(film_ids, test_size=0.2, random_state=42)

train_mask = df_long["titleId"].isin(train_films)
test_mask  = df_long["titleId"].isin(test_films)

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class FilmDataset(Dataset):
    def __init__(self, texts, labels, max_len=128):  # ← ΑΛΛΑΓΗ (ήταν 256)
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=max_len,
            return_tensors="pt"
        )
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

train_texts  = df_long[train_mask]["plotMedium"].fillna("").tolist()
train_labels = df_long[train_mask]["admissions_zscore"].tolist()
test_texts   = df_long[test_mask]["plotMedium"].fillna("").tolist()
test_labels  = df_long[test_mask]["admissions_zscore"].tolist()

train_dataset = FilmDataset(train_texts, train_labels)
test_dataset  = FilmDataset(test_texts,  test_labels)

In [ ]:
class FilmRegressor(torch.nn.Module):
    def __init__(self, model_name=MODEL_NAME):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.regressor = torch.nn.Sequential(
            torch.nn.Dropout(0.2),
            torch.nn.Linear(self.bert.config.hidden_size, 1)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_token = outputs.last_hidden_state[:, 0, :]
        return self.regressor(cls_token).squeeze(-1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = FilmRegressor().to(device)
print(f"Model ready on: {device}")

In [ ]:
EPOCHS     = 1          # ← 3 → 1 (αρκεί για ένα αποτέλεσμα)
BATCH_SIZE = 64         # ← 16 → 64 (4x λιγότερα βήματα)
LR         = 2e-5

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True                        # ← pin_memory για CPU speedup
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, num_workers=2, pin_memory=True
)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
loss_fn   = torch.nn.MSELoss()

# Τρέχει μόνο σε 10% του dataset — αρκεί για αποτέλεσμα
from itertools import islice
MAX_BATCHES = len(train_loader) // 10   # ← 10% των batches

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in islice(train_loader, MAX_BATCHES):   # ← ΑΛΛΑΓΗ
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        optimizer.zero_grad(set_to_none=True)          # ← ελαφρά πιο γρήγορο
        preds = model(input_ids, attention_mask)
        loss  = loss_fn(preds, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} — Train Loss: {total_loss/MAX_BATCHES:.4f}")

In [ ]:
import os
save_dir = PROJECT_ROOT / "outputs" / "distilbert_regressor"
os.makedirs(save_dir, exist_ok=True)
torch.save(model.state_dict(), save_dir / "model_weights.pt")

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        preds = model(input_ids, attention_mask)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

r2   = r2_score(all_labels, all_preds)
rmse = np.sqrt(mean_squared_error(all_labels, all_preds))

print(f"Test R²:   {r2:.4f}   (Juliusz Ridge baseline: ~0.113)")
print(f"Test RMSE: {rmse:.4f}")